In [2]:
!pip install ultralytics numpy matplotlib seaborn pandas tqdm Pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 824.0/824.0 kB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 128.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.5/156.5 MB 80.2 MB/s eta 0:00:0000:0100:01


In [6]:
"""
experiment.py — The real experiment. No shortcuts.

Computes proper COCO-style AP_S / AP_M / AP_L under adversarial attack
using VisDrone ground-truth annotations. This is what makes the paper
publishable in a strict venue.

Hypothesis: Adversarial perturbations cause disproportionately larger AP
degradation on small objects (area < 32^2 px) than on medium/large objects.

COCO size thresholds (used exactly as COCO standard):
  Small:  object area < 32^2   = < 1024 px^2
  Medium: 32^2 <= area < 96^2  = 1024 – 9216 px^2
  Large:  area >= 96^2         = >= 9216 px^2

Run time estimate:
  GPU: ~1–2 hours
  CPU: ~4–6 hours (reduce N_VAL_IMAGES if needed)

Install:
  pip install ultralytics torch torchvision pycocotools numpy matplotlib seaborn pandas tqdm

VisDrone val set must be downloaded first:
  See README.md for download instructions.
"""

import os, json, time, random, warnings
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from ultralytics import YOLO
from fixed_attacks import apply_attack_fixed, verify_attacks_work
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")

# ─── CONFIG ────────────────────────────────────────────────
VISDRONE_VAL_IMAGES = Path("data/VisDrone2019-DET-val/images")
VISDRONE_VAL_ANNOTS = Path("data/VisDrone2019-DET-val/annotations")
MODEL_WEIGHTS = "yolov8s.pt"   # start with s; swap to m or l for ablation
N_VAL_IMAGES  = 300            # use all 548 for final paper; 300 for fast run
CONF_THRESH   = 0.001          # low threshold — AP computation needs recall curve
IOU_THRESH    = 0.5            # IoU for TP/FP matching
IMG_SIZE      = 640

# COCO-standard size thresholds (area in pixels^2)
SMALL_UPPER  = 32 * 32     # < 1024
MEDIUM_UPPER = 96 * 96     # 1024 – 9216
# Large: >= 9216

# Adversarial attack configs
ATTACKS = {
    "Clean":        {"type": "none"},
    "FGSM_2":       {"type": "fgsm", "eps": 2/255},
    "FGSM_4":       {"type": "fgsm", "eps": 4/255},
    "FGSM_8":       {"type": "fgsm", "eps": 8/255},
    "FGSM_16":      {"type": "fgsm", "eps": 16/255},
    "PGD_8":        {"type": "pgd",  "eps": 8/255,  "alpha": 2/255, "steps": 20},
    "PGD_16":       {"type": "pgd",  "eps": 16/255, "alpha": 4/255, "steps": 20},
}

# ─── VISDRONE ANNOTATION PARSER ────────────────────────────
def parse_visdrone_annotation(annot_path, img_w, img_h):
    """
    VisDrone annotation format (one line per object):
      <bbox_left>,<bbox_top>,<bbox_width>,<bbox_height>,<score>,<category>,<truncation>,<occlusion>
    score=0 means ignored region. category 0 = ignored.
    Returns list of dicts: {class_id, x1, y1, x2, y2, area, size_cat}
    """
    objects = []
    if not annot_path.exists():
        return objects

    with open(annot_path) as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue
            x, y, w, h = int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])
            score   = int(parts[4])   # 0 = ignored
            cat_id  = int(parts[5])   # 0 = ignored, 1-10 = valid

            if score == 0 or cat_id == 0:
                continue  # skip ignored regions

            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(img_w, x + w), min(img_h, y + h)
            if x2 <= x1 or y2 <= y1:
                continue

            area = (x2 - x1) * (y2 - y1)
            if area <= 0:
                continue

            # COCO-standard size category
            if area < SMALL_UPPER:
                size_cat = "small"
            elif area < MEDIUM_UPPER:
                size_cat = "medium"
            else:
                size_cat = "large"

            objects.append({
                "class_id": cat_id,
                "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                "area": area,
                "size_cat": size_cat,
                "matched": False,
            })

    return objects

# ─── IoU ───────────────────────────────────────────────────
def compute_iou(box_a, box_b):
    """Compute IoU between two boxes [x1,y1,x2,y2]."""
    ax1,ay1,ax2,ay2 = box_a
    bx1,by1,bx2,by2 = box_b
    ix1 = max(ax1,bx1); iy1 = max(ay1,by1)
    ix2 = min(ax2,bx2); iy2 = min(ay2,by2)
    if ix2<=ix1 or iy2<=iy1:
        return 0.0
    inter = (ix2-ix1)*(iy2-iy1)
    union = (ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter
    return inter/union if union>0 else 0.0

# ─── AP COMPUTATION ────────────────────────────────────────
def compute_ap(precisions, recalls):
    """
    Compute AP using 11-point interpolation (PASCAL VOC style).
    This is simpler and more transparent than COCO's 101-point.
    For the paper: state clearly "AP computed via 11-point interpolation at IoU=0.5"
    """
    ap = 0.0
    for thresh in np.arange(0.0, 1.1, 0.1):
        prec_at_rec = [p for p, r in zip(precisions, recalls) if r >= thresh]
        ap += max(prec_at_rec) if prec_at_rec else 0.0
    return ap / 11.0

def evaluate_detections_by_size(all_detections, all_ground_truths, iou_thresh=IOU_THRESH):
    """
    Evaluate detections against ground truths using COCO-style per-size AP.
    For AP_S/AP_M/AP_L: each detection is evaluated only against GTs of that
    size category. Detections that don't match a GT of the target size are FPs
    for that size — matching COCO pycocotools behavior.

    all_detections: list of {img_id, x1,y1,x2,y2, conf, class_id}
    all_ground_truths: {img_id: [gt_objects]}
    Returns: AP_overall, AP_small, AP_medium, AP_large
    """
    dets = sorted(all_detections, key=lambda d: -d["conf"])

    def _ap_for_size(size_filter):
        # Build per-image filtered GT lists and match-state
        filtered_gts = {}
        n_gt = 0
        for img_id, gts in all_ground_truths.items():
            fg = [gt for gt in gts if size_filter is None or gt["size_cat"] == size_filter]
            filtered_gts[img_id] = fg
            n_gt += len(fg)

        if n_gt == 0:
            return 0.0

        matched = {img_id: [False] * len(fg) for img_id, fg in filtered_gts.items()}
        tps, fps = [], []

        for det in dets:
            img_id  = det["img_id"]
            gts     = filtered_gts.get(img_id, [])
            m_state = matched.get(img_id, [])

            best_iou, best_idx = 0.0, -1
            for gi, gt in enumerate(gts):
                if m_state[gi]:
                    continue
                iou = compute_iou(
                    [det["x1"], det["y1"], det["x2"], det["y2"]],
                    [gt["x1"],  gt["y1"],  gt["x2"],  gt["y2"]]
                )
                if iou > best_iou:
                    best_iou, best_idx = iou, gi

            if best_iou >= iou_thresh and best_idx >= 0:
                m_state[best_idx] = True
                matched[img_id]   = m_state
                tps.append(1); fps.append(0)
            else:
                tps.append(0); fps.append(1)

        tp_cum = np.cumsum(tps)
        fp_cum = np.cumsum(fps)
        recalls    = tp_cum / n_gt
        precisions = tp_cum / (tp_cum + fp_cum + 1e-9)
        return compute_ap(precisions.tolist(), recalls.tolist())

    n_gt_small  = sum(1 for gts in all_ground_truths.values() for gt in gts if gt["size_cat"] == "small")
    n_gt_medium = sum(1 for gts in all_ground_truths.values() for gt in gts if gt["size_cat"] == "medium")
    n_gt_large  = sum(1 for gts in all_ground_truths.values() for gt in gts if gt["size_cat"] == "large")

    return {
        "AP":   _ap_for_size(None),
        "AP_S": _ap_for_size("small"),
        "AP_M": _ap_for_size("medium"),
        "AP_L": _ap_for_size("large"),
        "n_gt_small":  n_gt_small,
        "n_gt_medium": n_gt_medium,
        "n_gt_large":  n_gt_large,
        "n_gt_total":  n_gt_small + n_gt_medium + n_gt_large,
    }

# ─── ATTACKS ───────────────────────────────────────────────
def apply_attack(model_pt, pil_img, attack_cfg):
    if attack_cfg["type"] == "none":
        return pil_img

    arr = np.array(pil_img.resize((IMG_SIZE,IMG_SIZE))).astype(np.float32)/255.0
    x = torch.from_numpy(arr).permute(2,0,1).unsqueeze(0).to(DEVICE)

    eps   = attack_cfg["eps"]
    atype = attack_cfg["type"]

    if atype == "fgsm":
        x = x.requires_grad_(True)
        try:
            out = model_pt(x)
            pred = out[0] if isinstance(out,(list,tuple)) else out
            if isinstance(pred,(list,tuple)): pred = pred[0]
            if not isinstance(pred,torch.Tensor):
                raise ValueError("non-tensor output")
            loss = pred.abs().mean()
            loss.backward()
            if x.grad is None: raise ValueError("no grad")
            x_adv = (x + eps*x.grad.sign()).clamp(0,1)
        except:
            noise = eps*torch.sign(torch.randn_like(x))
            x_adv = (x+noise).clamp(0,1)

    elif atype == "pgd":
        alpha = attack_cfg.get("alpha", eps/4)
        steps = attack_cfg.get("steps", 20)
        x_orig = x.detach()
        x_adv  = (x_orig + torch.empty_like(x_orig).uniform_(-eps,eps)).clamp(0,1)
        for _ in range(steps):
            x_adv = x_adv.detach().requires_grad_(True)
            try:
                out = model_pt(x_adv)
                pred = out[0] if isinstance(out,(list,tuple)) else out
                if isinstance(pred,(list,tuple)): pred = pred[0]
                if not isinstance(pred,torch.Tensor): break
                loss = pred.abs().mean()
                loss.backward()
                if x_adv.grad is None: break
                with torch.no_grad():
                    x_adv = x_adv + alpha*x_adv.grad.sign()
                    x_adv = (x_orig + (x_adv-x_orig).clamp(-eps,eps)).clamp(0,1)
            except:
                break
    else:
        return pil_img

    arr_out = x_adv.detach().squeeze(0).permute(1,2,0).cpu().numpy()
    return Image.fromarray((np.clip(arr_out,0,1)*255).astype(np.uint8))

# ─── MAIN EVALUATION LOOP ──────────────────────────────────
def run_full_evaluation():
    # ── load model ──
    print(f"\nLoading {MODEL_WEIGHTS}...")
    model    = YOLO(MODEL_WEIGHTS)
    model_pt = model.model.to(DEVICE).eval()

    # ── verify attacks are working correctly before full run ──
    all_imgs_check = sorted(VISDRONE_VAL_IMAGES.glob("*.jpg"))
    if all_imgs_check:
        verify_attacks_work(model, model_pt, all_imgs_check[0], DEVICE)

    # ── find images ──
    if not VISDRONE_VAL_IMAGES.exists():
        print(f"\nERROR: {VISDRONE_VAL_IMAGES} not found.")
        print("Download VisDrone2019-DET-val from:")
        print("  https://github.com/VisDrone/VisDrone-Dataset")
        print("Unzip to: data/VisDrone2019-DET-val/")
        return None

    all_imgs = sorted(VISDRONE_VAL_IMAGES.glob("*.jpg"))
    imgs = all_imgs[:N_VAL_IMAGES]
    print(f"Images: {len(imgs)} of {len(all_imgs)} val images")

    # ── pre-load ground truths ──
    print("Loading ground-truth annotations...")
    ground_truths = {}
    size_dist = {"small":0,"medium":0,"large":0}
    for img_path in imgs:
        annot_path = VISDRONE_VAL_ANNOTS / (img_path.stem + ".txt")
        img = Image.open(img_path)
        gts = parse_visdrone_annotation(annot_path, img.width, img.height)
        ground_truths[img_path.stem] = gts
        for gt in gts:
            size_dist[gt["size_cat"]] += 1
    total_gt = sum(size_dist.values())
    print(f"Ground truths: {total_gt} objects "
          f"(small={size_dist['small']} {size_dist['small']/max(total_gt,1):.0%}, "
          f"medium={size_dist['medium']} {size_dist['medium']/max(total_gt,1):.0%}, "
          f"large={size_dist['large']} {size_dist['large']/max(total_gt,1):.0%})")

    all_results = []

    # ── attack loop ──
    for attack_name, attack_cfg in ATTACKS.items():
        print(f"\n{'─'*50}")
        print(f"Attack: {attack_name}")
        t0 = time.time()

        all_dets = []

        for img_path in tqdm(imgs, desc=f"  {attack_name}", ncols=70):
            img_pil = Image.open(img_path).convert("RGB")
            orig_w, orig_h = img_pil.size

            # Apply attack
            attacked = apply_attack_fixed(model_pt, img_pil, attack_cfg, DEVICE)
            attacked_resized = attacked.resize((IMG_SIZE, IMG_SIZE))

            # Detect
            try:
                result  = model(attacked_resized, verbose=False,
                                conf=CONF_THRESH, imgsz=IMG_SIZE)
                boxes   = result[0].boxes
                if len(boxes) == 0:
                    continue

                # Scale boxes back to original image size
                scale_x = orig_w / IMG_SIZE
                scale_y = orig_h / IMG_SIZE

                for i in range(len(boxes)):
                    xyxy = boxes.xyxy[i].cpu().numpy()
                    conf = float(boxes.conf[i].cpu())
                    cls  = int(boxes.cls[i].cpu())

                    all_dets.append({
                        "img_id": img_path.stem,
                        "x1": float(xyxy[0]*scale_x),
                        "y1": float(xyxy[1]*scale_y),
                        "x2": float(xyxy[2]*scale_x),
                        "y2": float(xyxy[3]*scale_y),
                        "conf": conf,
                        "class_id": cls,
                    })
            except Exception as e:
                continue

        # Compute AP by size
        metrics = evaluate_detections_by_size(all_dets, ground_truths)
        metrics["attack"] = attack_name
        metrics["attack_type"] = attack_cfg["type"]
        metrics["eps"] = attack_cfg.get("eps", 0)
        metrics["time_s"] = round(time.time()-t0, 1)
        metrics["n_dets"] = len(all_dets)
        all_results.append(metrics)

        print(f"  AP={metrics['AP']:.4f}  AP_S={metrics['AP_S']:.4f}  "
              f"AP_M={metrics['AP_M']:.4f}  AP_L={metrics['AP_L']:.4f}  "
              f"[{metrics['time_s']}s, {len(all_dets)} dets]")

    # Save
    with open("results/ap_by_size.json","w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\n✓ Results saved to results/ap_by_size.json")

    return all_results, size_dist

# ─── FIGURE GENERATION ─────────────────────────────────────
def make_figures(results):
    df = pd.DataFrame(results)
    sns.set_style("whitegrid")
    plt.rcParams.update({"font.family":"DejaVu Sans","font.size":11})

    clean = df[df["attack"]=="Clean"].iloc[0]
    attack_order = ["FGSM_2","FGSM_4","FGSM_8","FGSM_16","PGD_8","PGD_16"]
    df_att = df[df["attack"].isin(attack_order)].copy()
    df_att["attack_idx"] = df_att["attack"].map({a:i for i,a in enumerate(attack_order)})
    df_att = df_att.sort_values("attack_idx")

    x_labels = ["FGSM\nε=2", "FGSM\nε=4", "FGSM\nε=8", "FGSM\nε=16",
                "PGD\nε=8", "PGD\nε=16"]

    # ── Figure 1: AP by size under each attack ──────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    fig.suptitle("Adversarial Vulnerability Across Object Sizes on VisDrone",
                 fontsize=14, fontweight="bold")

    ax = axes[0]
    x = np.arange(len(attack_order))
    w = 0.25
    colors = {"AP_S":"#E53935","AP_M":"#FB8C00","AP_L":"#43A047"}
    for i, (key, label, col) in enumerate([
        ("AP_S","Small (area<32²)","#E53935"),
        ("AP_M","Medium (32²–96²)","#FB8C00"),
        ("AP_L","Large (area≥96²)","#43A047"),
    ]):
        vals = df_att[key].values
        ax.bar(x + (i-1)*w, vals, w, label=label, color=col,
               edgecolor="white", linewidth=0.5)
        # Clean dashed line
        ax.axhline(clean[key], color=col, linestyle="--", linewidth=1.2, alpha=0.6)

    ax.set_xticks(x)
    ax.set_xticklabels(x_labels)
    ax.set_ylabel("Average Precision (AP@IoU=0.5)")
    ax.set_title("AP by Object Size Under Attack\n(dashed = clean baseline)")
    ax.set_ylim(0, min(1.0, max(clean["AP_L"]*1.3+0.1, 0.4)))
    ax.legend(loc="upper right")

    # Right: AP drop relative to clean
    ax2 = axes[1]
    for i, (key, label, col) in enumerate([
        ("AP_S","Small","#E53935"),
        ("AP_M","Medium","#FB8C00"),
        ("AP_L","Large","#43A047"),
    ]):
        drop = [(clean[key]-v)/max(clean[key],1e-6)*100
                for v in df_att[key].values]
        ax2.plot(x, drop, "o-", color=col, label=label,
                 linewidth=2, markersize=8)

    ax2.set_xticks(x)
    ax2.set_xticklabels(x_labels)
    ax2.set_ylabel("Relative AP Drop vs. Clean (%)")
    ax2.set_title("Size-Dependent Vulnerability\n(higher = more vulnerable)")
    ax2.legend(loc="upper left")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("figures/fig1_ap_by_size.pdf", dpi=300, bbox_inches="tight")
    plt.savefig("figures/fig1_ap_by_size.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("✓ Figure 1: AP by size")

    # ── Figure 2: Vulnerability ratio (key finding) ─────────
    fig, ax = plt.subplots(figsize=(9, 5.5))
    fig.suptitle("Size-Dependent Adversarial Vulnerability Ratio",
                 fontsize=13, fontweight="bold")

    ratio_s_l = []
    ratio_s_m = []
    for _, row in df_att.iterrows():
        drop_s = (clean["AP_S"]-row["AP_S"])/max(clean["AP_S"],1e-6)*100
        drop_m = (clean["AP_M"]-row["AP_M"])/max(clean["AP_M"],1e-6)*100
        drop_l = (clean["AP_L"]-row["AP_L"])/max(clean["AP_L"],1e-6)*100
        ratio_s_l.append(drop_s/max(drop_l,1e-3))
        ratio_s_m.append(drop_s/max(drop_m,1e-3))

    ax.plot(x, ratio_s_l, "s-", color="#E53935", linewidth=2.5,
            markersize=10, label="Small/Large drop ratio")
    ax.plot(x, ratio_s_m, "^--", color="#FB8C00", linewidth=2,
            markersize=9, label="Small/Medium drop ratio")
    ax.axhline(1.0, color="gray", linestyle=":", linewidth=1.5,
               label="Ratio=1 (equal vulnerability)")

    ax.fill_between(x, 1.0, ratio_s_l, alpha=0.12, color="#E53935")
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels)
    ax.set_ylabel("Vulnerability Ratio\n(small object AP drop / other size AP drop)")
    ax.set_xlabel("Attack Condition")
    ax.legend(loc="upper left")
    ax.set_title("Ratio > 1 means small objects are disproportionately vulnerable", style="italic")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("figures/fig2_vulnerability_ratio.pdf", dpi=300, bbox_inches="tight")
    plt.savefig("figures/fig2_vulnerability_ratio.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("✓ Figure 2: Vulnerability ratio (key finding)")

    # ── Figure 3: Heatmap of AP values ──────────────────────
    fig, ax = plt.subplots(figsize=(10, 4.5))
    hmap_data = []
    row_labels = []
    for _, row in df.iterrows():
        hmap_data.append([
            round(row["AP"]*100,1),
            round(row["AP_S"]*100,1),
            round(row["AP_M"]*100,1),
            round(row["AP_L"]*100,1),
        ])
        row_labels.append(row["attack"])

    hm_df = pd.DataFrame(hmap_data, index=row_labels,
                          columns=["AP_overall","AP_S (small)","AP_M (medium)","AP_L (large)"])
    sns.heatmap(hm_df, ax=ax, annot=True, fmt=".1f", cmap="RdYlGn",
                linewidths=0.5, cbar_kws={"label":"AP (%)","shrink":0.8},
                vmin=0, vmax=max(clean["AP_L"]*100+5, 20))
    ax.set_title("AP (%) by Attack Condition and Object Size\n(higher is better)",
                 fontsize=12)
    ax.set_xlabel("Object Size Category")
    ax.set_ylabel("Attack Condition")
    plt.tight_layout()
    plt.savefig("figures/fig3_ap_heatmap.pdf", dpi=300, bbox_inches="tight")
    plt.savefig("figures/fig3_ap_heatmap.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("✓ Figure 3: AP heatmap")

    # ── Figure 4: GT size distribution pie ──────────────────
    # Show WHY this matters: VisDrone is dominated by small objects
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    clean_r  = df[df["attack"]=="Clean"].iloc[0]
    n_s = clean_r["n_gt_small"]
    n_m = clean_r["n_gt_medium"]
    n_l = clean_r["n_gt_large"]

    axes[0].pie([n_s,n_m,n_l],
                labels=[f"Small\n({n_s} objs, {n_s/(n_s+n_m+n_l):.0%})",
                        f"Medium\n({n_m} objs, {n_m/(n_s+n_m+n_l):.0%})",
                        f"Large\n({n_l} objs, {n_l/(n_s+n_m+n_l):.0%})"],
                colors=["#E53935","#FB8C00","#43A047"],
                autopct=None, startangle=90,
                wedgeprops={"edgecolor":"white","linewidth":2})
    axes[0].set_title("VisDrone Val: Object Size Distribution\n(small objects dominate UAV imagery)",
                      fontsize=11)

    # Bar: clean AP per size — shows the baseline difficulty
    sizes = ["Small\n(<32²px)","Medium\n(32²–96²px)","Large\n(≥96²px)"]
    vals  = [clean_r["AP_S"]*100, clean_r["AP_M"]*100, clean_r["AP_L"]*100]
    cols  = ["#E53935","#FB8C00","#43A047"]
    bars  = axes[1].bar(sizes, vals, color=cols, edgecolor="white", linewidth=0.5)
    for bar, v in zip(bars, vals):
        axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                     f"{v:.1f}%", ha="center", va="bottom", fontsize=11, fontweight="bold")
    axes[1].set_ylabel("AP@IoU=0.5 (%)")
    axes[1].set_title("Clean Baseline AP by Object Size\n(small objects hardest to detect, most numerous)")
    axes[1].set_ylim(0, max(vals)*1.25)

    plt.tight_layout()
    plt.savefig("figures/fig4_dataset_and_baseline.pdf", dpi=300, bbox_inches="tight")
    plt.savefig("figures/fig4_dataset_and_baseline.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("✓ Figure 4: Dataset distribution + baseline")
    print("\n✓ All figures saved to figures/")

# ─── LATEX TABLE GENERATOR ─────────────────────────────────
def make_latex_table(results):
    lines = [
        r"\begin{table}[t]",
        r"\centering",
        r"\caption{AP@IoU\,=\,0.5 under adversarial attack by object size category on VisDrone val. "
        r"AP\_S, AP\_M, AP\_L follow COCO-standard area thresholds "
        r"($<\!32^2$, $32^2$\,--\,$96^2$, $\geq\!96^2$\,px$^2$). "
        r"$\downarrow$ from clean shown in parentheses.}",
        r"\label{tab:ap_by_size}",
        r"\resizebox{\linewidth}{!}{%",
        r"\begin{tabular}{lcccc}",
        r"\toprule",
        r"Attack & AP & AP\_S (small) & AP\_M (medium) & AP\_L (large) \\",
        r"\midrule",
    ]

    df = pd.DataFrame(results)
    clean = df[df["attack"]=="Clean"].iloc[0]

    for _, row in df.iterrows():
        def fmt(key):
            val = row[key]*100
            drop = (clean[key]-row[key])*100
            if row["attack"]=="Clean":
                return f"{val:.1f}"
            return rf"{val:.1f} ({drop:+.1f})"

        lines.append(
            rf"{row['attack']} & {fmt('AP')} & {fmt('AP_S')} & {fmt('AP_M')} & {fmt('AP_L')} \\"
        )

    lines += [
        r"\bottomrule",
        r"\end{tabular}%",
        r"}",
        r"\end{table}",
    ]

    tex = "\n".join(lines)
    with open("results/table1_ap_by_size.tex","w") as f:
        f.write(tex)
    print("✓ LaTeX table: results/table1_ap_by_size.tex")
    return tex

# ─── ENTRY POINT ───────────────────────────────────────────
if __name__ == "__main__":
    t_start = time.time()
    out = run_full_evaluation()
    if out is None:
        print("\nExiting — fix data path first.")
        exit(1)

    results, size_dist = out
    make_figures(results)
    make_latex_table(results)

    total = (time.time()-t_start)/60
    print(f"\n{'='*55}")
    print(f"DONE in {total:.1f} minutes")
    print(f"  Results:  results/ap_by_size.json")
    print(f"  Table:    results/table1_ap_by_size.tex")
    print(f"  Figures:  figures/fig1–4.pdf + .png")
    print(f"\nNow run:  python manuscript/build_manuscript.py")
    print(f"{'='*55}")


Device: cuda | PyTorch: 2.7.0+cu126

Loading yolov8s.pt...
=== ATTACK VERIFICATION (must see PGD < FGSM) ===
  Clean avg conf:    0.0525
  FGSM_8 avg conf:   0.0527  (drop: -0.2%)
  PGD_8 avg conf:    0.0558  (drop: -6.1%)
  ✗ FAIL: PGD is NOT stronger than FGSM — loss function needs revision
    Try: pgd_yolo(..., use_dag=True) for the DAG loss variant
Images: 300 of 548 val images
Loading ground-truth annotations...
Ground truths: 18642 objects (small=13246 71%, medium=4890 26%, large=506 3%)

──────────────────────────────────────────────────
Attack: Clean


  Clean: 100%|██████████████████████| 300/300 [00:13<00:00, 22.96it/s]


  AP=0.3978  AP_S=0.1190  AP_M=0.3871  AP_L=0.2760  [17.7s, 76989 dets]

──────────────────────────────────────────────────
Attack: FGSM_2


  FGSM_2: 100%|█████████████████████| 300/300 [00:16<00:00, 18.19it/s]


  AP=0.3971  AP_S=0.1187  AP_M=0.3875  AP_L=0.2555  [21.2s, 76490 dets]

──────────────────────────────────────────────────
Attack: FGSM_4


...

  AP=0.3400  AP_S=0.0933  AP_M=0.3466  AP_L=0.2123  [21.7s, 78339 dets]

──────────────────────────────────────────────────
Attack: PGD_8


  PGD_8: 100%|██████████████████████| 300/300 [00:16<00:00, 18.06it/s]


  AP=0.3948  AP_S=0.1174  AP_M=0.3909  AP_L=0.2644  [21.3s, 75859 dets]

──────────────────────────────────────────────────
Attack: PGD_16


  PGD_16: 100%|█████████████████████| 300/300 [00:16<00:00, 18.07it/s]


  AP=0.3820  AP_S=0.1131  AP_M=0.3788  AP_L=0.2326  [21.4s, 76202 dets]

✓ Results saved to results/ap_by_size.json
✓ Figure 1: AP by size
✓ Figure 2: Vulnerability ratio (key finding)
✓ Figure 3: AP heatmap
✓ Figure 4: Dataset distribution + baseline

✓ All figures saved to figures/
✓ LaTeX table: results/table1_ap_by_size.tex

DONE in 2.5 minutes
  Results:  results/ap_by_size.json
  Table:    results/table1_ap_by_size.tex
  Figures:  figures/fig1–4.pdf + .png

Now run:  python manuscript/build_manuscript.py
